# Phase 5: Generate and Lock Group Stage Predictions

This notebook generates predictions for all 72 group-stage matches.

**CRITICAL:** Once this CSV is saved, do NOT modify it. The calibration analysis depends on predictions being locked before June 11.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '../src')
from poisson_model import (
    load_and_weight_data,
    build_team_index,
    fit_poisson_model,
    predict_match
)

## Step 1: Load and fit the model (reuse from Phase 3)

In [2]:
# Load data and fit the Poisson model
df = load_and_weight_data('../data/processed/matches_filtered.csv')
teams, team_to_idx, idx_to_team = build_team_index(df)
n_teams = len(teams)

print(f"Fitting model on {len(df)} matches and {n_teams} teams...")
mu, alphas, betas = fit_poisson_model(df, team_to_idx, n_teams)
print(f"\nModel fit complete.")
print(f"mu = {mu:.4f}")
print(f"alpha: mean={np.mean(alphas):.4f}, std={np.std(alphas):.4f}")
print(f"beta: mean={np.mean(betas):.4f}, std={np.std(betas):.4f}")

INFO:poisson_model:Loaded 1433 matches. Weight distribution:
weight
0.5    1433
Name: count, dtype: int64
INFO:poisson_model:Found 48 unique teams
INFO:poisson_model:Starting MLE optimization...


Fitting model on 1433 matches and 48 teams...


INFO:poisson_model:Optimization complete. Success: True, NLL: 2031.58
INFO:poisson_model:After normalization: mu=1.2189, alpha_mean=1.0406, beta_mean=1.0306
INFO:poisson_model:After clipping: mu=1.2189, alpha range=[0.59, 1.70], beta range=[0.61, 1.74]



Model fit complete.
mu = 1.2189
alpha: mean=1.0406, std=0.2966
beta: mean=1.0306, std=0.2568


## Step 2: Load fixture list

In [8]:
# Load the fixture list
fixtures = pd.read_csv('../data/raw/wc_2026_fixtures.csv')

# Filter to group stage only (rows where 'group' column contains 'Group')
group_stage = fixtures[fixtures['group'].str.contains('Group', na=False)].copy()
group_stage = group_stage.reset_index(drop=True)

print(f"Total fixtures: {len(fixtures)}")
print(f"Group stage matches: {len(group_stage)}")
print(f"\nFirst 5 matches:")
print(group_stage[['date', 'group', 'home_team', 'away_team']].head())

Total fixtures: 104
Group stage matches: 72

First 5 matches:
         date    group       home_team               away_team
0  2026-06-11  Group A          Mexico            South Africa
1  2026-06-12  Group A  Korea Republic                 Czechia
2  2026-06-12  Group B          Canada  Bosnia and Herzegovina
3  2026-06-12  Group D             USA                Paraguay
4  2026-06-13  Group C           Haiti                Scotland


## Step 3: Generate predictions for all group stage matches

In [12]:
# Fix team name mismatches between fixture file and dataset
team_name_map = {
    'USA': 'United States',
    'Turkey': 'Türkiye',
    "Cote d'Ivoire": "Côte d'Ivoire",
    'Curacao': 'Curaçao',
    'Cape Verde': 'Cabo Verde',
    'Iran': 'IR Iran',
    'Congo DR': 'DR Congo'
}

# Apply mapping to home and away teams
group_stage['home_team'] = group_stage['home_team'].replace(team_name_map)
group_stage['away_team'] = group_stage['away_team'].replace(team_name_map)

print("Fixed team names in fixtures:")
print(group_stage[['home_team', 'away_team']].head(10))

Fixed team names in fixtures:
        home_team               away_team
0          Mexico            South Africa
1  Korea Republic                 Czechia
2          Canada  Bosnia and Herzegovina
3   United States                Paraguay
4           Haiti                Scotland
5       Australia                 Türkiye
6          Brazil                 Morocco
7           Qatar             Switzerland
8   Côte d'Ivoire                 Ecuador
9         Germany                 Curaçao


In [13]:
predictions = []
failed_matches = []

for idx, row in group_stage.iterrows():
    team_a = row['home_team']
    team_b = row['away_team']
    
    try:
        pred = predict_match(team_a, team_b, mu, alphas, betas, teams, team_to_idx)
        
        # Determine predicted winner
        if pred['p_win_a'] > pred['p_win_b']:
            predicted_winner = team_a
        elif pred['p_win_b'] > pred['p_win_a']:
            predicted_winner = team_b
        else:
            predicted_winner = 'draw'
        
        top_scoreline = pred['top_scorelines'][0]
        
        predictions.append({
            'match_id': idx + 1,
            'date': row['date'],
            'group': row['group'],
            'home_team': team_a,
            'away_team': team_b,
            'p_home_win': pred['p_win_a'],
            'p_draw': pred['p_draw'],
            'p_away_win': pred['p_win_b'],
            'predicted_winner': predicted_winner,
            'top_scoreline': f"{int(top_scoreline[0])}-{int(top_scoreline[1])}",
            'location': row['location']
        })
    except KeyError as e:
        failed_matches.append((team_a, team_b, str(e)))
        print(f"ERROR: Could not find {e} for match {team_a} vs {team_b}")

print(f"\nGenerated predictions for {len(predictions)} matches")
if failed_matches:
    print(f"Failed to generate predictions for {len(failed_matches)} matches:")
    for a, b, err in failed_matches:
        print(f"  {a} vs {b}: {err}")


Generated predictions for 72 matches


## Step 4: Create predictions dataframe and inspect

In [14]:
predictions_df = pd.DataFrame(predictions)

print(f"Predictions dataframe shape: {predictions_df.shape}")
print(f"\nColumns: {list(predictions_df.columns)}")
print(f"\nFirst 10 predictions:")
print(predictions_df[['date', 'group', 'home_team', 'away_team', 'p_home_win', 'p_draw', 'p_away_win', 'predicted_winner']].head(10))

Predictions dataframe shape: (72, 11)

Columns: ['match_id', 'date', 'group', 'home_team', 'away_team', 'p_home_win', 'p_draw', 'p_away_win', 'predicted_winner', 'top_scoreline', 'location']

First 10 predictions:
         date    group       home_team               away_team  p_home_win  \
0  2026-06-11  Group A          Mexico            South Africa    0.494027   
1  2026-06-12  Group A  Korea Republic                 Czechia    0.420968   
2  2026-06-12  Group B          Canada  Bosnia and Herzegovina    0.364318   
3  2026-06-12  Group D   United States                Paraguay    0.428783   
4  2026-06-13  Group C           Haiti                Scotland    0.226151   
5  2026-06-13  Group D       Australia                 Türkiye    0.298259   
6  2026-06-13  Group C          Brazil                 Morocco    0.521158   
7  2026-06-13  Group B           Qatar             Switzerland    0.170949   
8  2026-06-14  Group E   Côte d'Ivoire                 Ecuador    0.331195   
9  202

## Step 5: Sanity checks

In [15]:
# Check that probabilities sum to 1.0
prob_sums = predictions_df['p_home_win'] + predictions_df['p_draw'] + predictions_df['p_away_win']
print(f"Probability sums:")
print(f"  Mean: {prob_sums.mean():.6f}")
print(f"  Min: {prob_sums.min():.6f}")
print(f"  Max: {prob_sums.max():.6f}")
print(f"  All close to 1.0: {np.allclose(prob_sums, 1.0)}")

# Distribution of predicted winners
print(f"\nPredicted winner distribution:")
print(predictions_df['predicted_winner'].value_counts())

# Show some interesting matchups
print(f"\n=== Selected Group Stage Predictions ===")
for _, row in predictions_df.iterrows():
    if row['home_team'] in ['France', 'Brazil', 'Argentina', 'England', 'Spain', 'Germany']:
        print(f"{row['date']} | {row['group']:8s} | {row['home_team']:20s} {row['p_home_win']:.3f} vs {row['p_away_win']:.3f} {row['away_team']:20s} (draw: {row['p_draw']:.3f})")

Probability sums:
  Mean: 0.999970
  Min: 0.999187
  Max: 1.000000
  All close to 1.0: False

Predicted winner distribution:
predicted_winner
Mexico                    3
United States             3
Brazil                    3
Switzerland               3
Germany                   3
Netherlands               3
Spain                     3
Belgium                   3
France                    3
Argentina                 3
England                   3
Portugal                  3
Korea Republic            2
Canada                    2
Türkiye                   2
Ecuador                   2
Uruguay                   2
IR Iran                   2
Austria                   2
Colombia                  2
Morocco                   2
Japan                     2
Senegal                   2
Croatia                   2
Scotland                  1
Tunisia                   1
Norway                    1
Ghana                     1
Czechia                   1
Egypt                     1
Algeria           

## Step 6: Save predictions (LOCKED FOR TOURNAMENT)

In [16]:
# Create predictions directory if it doesn't exist
import os
os.makedirs('../predictions', exist_ok=True)

# Save predictions
output_path = '../predictions/group_stage_predictions.csv'
predictions_df.to_csv(output_path, index=False)

print(f"✓ Saved {len(predictions_df)} group stage predictions to {output_path}")
print(f"\n!!! LOCKED FOR TOURNAMENT !!!")
print(f"Do NOT modify this file after June 11, 2026.")
print(f"Commit to git: git add predictions/group_stage_predictions.csv && git commit -m 'Phase 5: Locked group stage predictions'")

✓ Saved 72 group stage predictions to ../predictions/group_stage_predictions.csv

!!! LOCKED FOR TOURNAMENT !!!
Do NOT modify this file after June 11, 2026.
Commit to git: git add predictions/group_stage_predictions.csv && git commit -m 'Phase 5: Locked group stage predictions'
